
**ARCH 594/508: AI Seminar/Studio**  
Spring 2026  

**Instructor:** Narjes Abbasabadi, Ph.D. Assistant Professor  
Email: nabbasab@uw.edu  

**TA:** Hasib Hossain, MSDT Student

Email: hasib97@uw.edu


Department of Architecture  
College of Built Environments  
University of Washington  
Website: [https://silab.be.uw.edu](https://silab.be.uw.edu)

## Generative AI in Architecture: An Overview


In this notebook we will explore how **generative AI** is being used in architectural design. Rather than focusing on a single model, we will survey the landscape of generative AI tools and demonstrate several techniques that architects are beginning to adopt in practice.

We will cover:
- **Text-to-image generation** using diffusion models (Stable Diffusion)
- **Neural style transfer** for architectural visualization
- **Image captioning** to analyze and describe architectural images using AI
- **Large Language Models (LLMs)** for generating design briefs and program descriptions
- **GANs and floorplan generation** as a conceptual overview

- The [Generative AI in Architectural Design](https://www.sciencedirect.com/science/article/abs/pii/S0926580525002146) review paper by Jang et al. provides a comprehensive overview of the field
- The [Conversational AI-Enhanced Design Process](https://doi.org/10.1007/s44223-025-00092-5) paper by Cheung et al. explores AI as a design partner

## What is Generative AI?

Generative AI refers to models that can **create new content** (images, text, 3D models, music) rather than just classifying or predicting from existing data. In architecture, generative AI is being used across the design process:

**Early Design / Concept Phase:**
- Generating concept images from text descriptions
- Exploring multiple design directions rapidly
- Mood boards and visual references

**Schematic Design:**
- Floorplan generation from adjacency requirements
- Massing studies and form exploration
- Style transfer to visualize design intent

**Design Development:**
- Performance-informed design optimization
- Material and detail visualization
- Rendering and photorealistic previews

**Communication and Analysis:**
- AI-generated descriptions of design intent
- Automated image captioning for documentation
- Sentiment analysis of built environment conditions

### Key Model Types

| Model Type | How It Works | Architecture Application |
|---|---|---|
| **Diffusion Models** (Stable Diffusion, DALL-E, Midjourney) | Start from noise, iteratively denoise guided by text | Concept visualization, rendering |
| **GANs** (Generative Adversarial Networks) | Generator vs. discriminator compete to produce realistic outputs | Floorplan generation, facade design |
| **VAEs** (Variational Autoencoders) | Encode data to a latent space, then decode new samples | Layout interpolation, design space exploration |
| **LLMs** (GPT, Claude, Gemini) | Predict next tokens in a sequence | Design briefs, code generation, critique |
| **Multimodal Models** (GPT-4V, BLIP, LLaVA) | Process both images and text together | Image analysis, design feedback |

## 0. Setup and Installation

First lets install the libraries we will need. This notebook uses a **T4 GPU** runtime for the image generation sections.

To enable GPU in Colab:
1. Go to **Runtime > Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**

Then run the install cell below. **After it finishes, you must restart the runtime** by clicking **Runtime > Restart session**. This is required because upgrading Pillow takes effect only after a restart. After restarting, **skip the install cell** and continue running from the imports cell onward.

In [ ]:
## NOTE: RUN THIS CELL ONCE, THEN RESTART THE RUNTIME
## After restart, skip this cell and continue from the imports cell below

# Pin Pillow to a version compatible with both diffusers and gradio
!pip install -Uq "Pillow>=10.0,<12.0"

# Install AI packages with compatible versions
!pip install -Uq diffusers transformers accelerate safetensors

# Pin TensorFlow to match tensorflow-decision-forests requirement
!pip install -Uq tensorflow==2.19.0 tensorflow-hub

!pip install -Uq matplotlib

print()
print("=" * 60)
print("INSTALLATION COMPLETE")
print("=" * 60)
print("Now go to: Runtime > Restart session")
print("After restart, SKIP this cell and run from imports onward.")
print("=" * 60)

In [ ]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Helper function to display images side by side
def show_images(images, titles=None, figsize=(16, 5)):
    """Display a list of PIL images or numpy arrays side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, img in enumerate(images):
        axes[i].imshow(img)
        axes[i].axis('off')
        if titles:
            axes[i].set_title(titles[i], fontsize=12)
    plt.tight_layout()
    plt.show()

# Helper function to load an image from a URL
import urllib.request

def load_image_from_url(url, size=None):
    """Download an image from a URL and return it as a PIL Image."""
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    img = Image.open(BytesIO(response.read())).convert('RGB')
    if size:
        img = img.resize(size)
    return img

print("Helper functions loaded.")

---

# Part 1: Text-to-Image Generation with Diffusion Models

Text-to-image models are the most visible application of generative AI in architecture today. Tools like **Stable Diffusion**, **Midjourney**, and **DALL-E** allow designers to generate photorealistic architectural visualizations from text descriptions alone.

### How Diffusion Models Work

1. The model starts with **pure noise** (random pixels)
2. A text encoder converts your prompt into a numerical representation
3. The model **iteratively denoises** the image, guided by the text embedding
4. After many steps, a coherent image emerges

The quality of the output depends heavily on **prompt engineering** — how you describe what you want.

We will use the **Stable Diffusion** model through the Hugging Face `diffusers` library.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# Load a lightweight Stable Diffusion model
# We use the v1.5 model for faster loading on free Colab GPUs
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to("cuda")

# Reduce memory usage
pipe.enable_attention_slicing()

print("Stable Diffusion model loaded successfully.")

### Generate an Architectural Image

Lets generate our first image. Notice how the prompt includes specific architectural vocabulary: building type, materials, style, lighting, and quality modifiers.

In [ ]:
# Generate an architectural image from a text prompt
prompt = "A modern museum with a cantilevered concrete facade, large glass walls, reflecting pool, golden hour lighting, architectural photography, 8K"

# Set a seed for reproducibility
generator = torch.Generator("cuda").manual_seed(498)

image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]

show_images([image], titles=["Generated: Modern Museum"])

### Comparing Different Architectural Prompts

One of the most powerful uses of text-to-image AI in architecture is **rapid concept exploration**. Lets generate three different building types from different prompts in a single run.

In [ ]:
# Generate three different architectural concepts
prompts = [
    "A brutalist concrete apartment building, raw exposed concrete, geometric balconies, overcast sky, architectural photography",
    "A sustainable bamboo pavilion in a tropical forest, open air structure, thatched roof, dappled sunlight, architectural photography",
    "A futuristic glass skyscraper with a twisting form, reflective facade, city skyline, blue sky, architectural photography",
]

images = []
for i, p in enumerate(prompts):
    generator = torch.Generator("cuda").manual_seed(498 + i)
    img = pipe(p, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]
    images.append(img)

show_images(images, titles=["Brutalist", "Sustainable Bamboo", "Futuristic Glass"])

### The Role of Prompt Engineering

The quality and relevance of generated images depends on how well you craft the prompt. Here are key strategies for architectural prompts:

| Element | Examples | Why It Matters |
|---|---|---|
| **Building type** | museum, residence, chapel, tower | Anchors the overall form |
| **Style/movement** | brutalist, parametric, minimalist, vernacular | Sets the design language |
| **Materials** | concrete, timber, glass, corten steel, rammed earth | Controls texture and color |
| **Lighting** | golden hour, overcast, twilight, morning mist | Sets mood and atmosphere |
| **Context** | hillside, urban plaza, waterfront, desert | Grounds the building in a site |
| **Quality keywords** | architectural photography, 8K, photorealistic | Pushes output toward realism |
| **Negative prompts** | cartoon, blurry, distorted, low quality | Removes unwanted qualities |

### The Effect of Guidance Scale

The `guidance_scale` parameter controls how closely the model follows your prompt. Lets compare three values.

In [ ]:
# Compare guidance scale values
prompt = "A rammed earth house in the desert with a green courtyard, flat roof, warm evening light, architectural photography, 8K"

guidance_values = [3.0, 7.5, 12.0]
images = []
for g in guidance_values:
    generator = torch.Generator("cuda").manual_seed(498)
    img = pipe(prompt, num_inference_steps=30, guidance_scale=g, generator=generator).images[0]
    images.append(img)

show_images(images, titles=[f"guidance={g}" for g in guidance_values])

- **Low guidance (3.0):** More creative and loose, the model takes liberties with interpretation
- **Medium guidance (7.5):** Balanced, the default setting for most use cases
- **High guidance (12.0):** Strictly follows the prompt, but may introduce visual artifacts

For most architectural visualization, values between 5.0 and 9.0 work best.

---

# Part 2: Neural Style Transfer for Architectural Visualization

Neural style transfer takes the **content** of one image and renders it in the **style** of another. For architects, this is useful for:
- Applying a watercolor or sketch aesthetic to a 3D render
- Visualizing how a building might look if it adopted the material palette of another project
- Creating presentation-quality mood images

We will use the **Arbitrary Image Stylization** model from TensorFlow Hub, which can apply any style to any content image in real time.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# Load the arbitrary style transfer model from TensorFlow Hub
style_transfer_model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')

print("Style transfer model loaded successfully.")

In [ ]:
# Helper function to preprocess images for the style transfer model
def preprocess_for_style_transfer(image, max_dim=512):
    """Resize and normalize an image for the style transfer model."""
    img = np.array(image).astype(np.float32) / 255.0
    shape = img.shape[:2]
    scale = max_dim / max(shape)
    new_shape = (int(shape[0] * scale), int(shape[1] * scale))
    img = tf.image.resize(img, new_shape)
    img = img[tf.newaxis, :]
    return img

print("Preprocessing function ready.")

### Load a Content Image and a Style Image

We will download a photo of a building (content) and a painting or texture (style), then combine them.

In [ ]:
# Load a content image (a building photo) and a style image (a painting/texture)
# You can replace these URLs with your own images

# Download images using urllib (more reliable than requests for some hosts)
import urllib.request

# Content: a modern building photograph
content_url = "https://images.unsplash.com/photo-1486325212027-8081e485255e?w=640"
# Style: a painting texture
style_url = "https://images.unsplash.com/photo-1579783902614-a3fb3927b6a5?w=640"

def download_image(url):
    """Download an image from a URL with proper headers."""
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    img = Image.open(BytesIO(response.read())).convert('RGB')
    return img

content_image = download_image(content_url)
style_image = download_image(style_url)

show_images([content_image, style_image], titles=["Content: Building Photo", "Style: Painting"])

In [ ]:
# Apply style transfer
content_tensor = preprocess_for_style_transfer(content_image)
style_tensor = preprocess_for_style_transfer(style_image)

# Run the model
stylized = style_transfer_model(content_tensor, style_tensor)[0]

# Convert result back to displayable image
stylized_image = np.squeeze(stylized.numpy())
stylized_image = np.clip(stylized_image, 0, 1)

show_images(
    [content_image, style_image, (stylized_image * 255).astype(np.uint8)],
    titles=["Content", "Style", "Stylized Result"],
    figsize=(18, 6)
)

### Try Your Own Style Transfer

**Exercise:** Replace the `content_url` and `style_url` in the cell above with your own images. Some ideas:
- Use a photo of a building you are studying as the content
- Use a sketch, watercolor, or architectural drawing as the style
- Try using a material texture (brick, wood grain, marble) as the style image

This technique can be useful for creating presentation visuals that have a consistent aesthetic language.

---

# Part 3: AI Image Understanding and Captioning

Generative AI is not only about creating images; it can also **understand and describe** them. Image captioning models can analyze architectural photos and generate text descriptions. This is useful for:

- Automated documentation of design precedents
- Accessibility (describing buildings for visually impaired users)
- Searching large image databases by content
- Evaluating what a model "sees" in a design

We will use the **BLIP** (Bootstrapped Language-Image Pre-training) model from Salesforce, which can both caption images and answer questions about them.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load the BLIP image captioning model
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")

print("BLIP captioning model loaded successfully.")

In [ ]:
# Caption the building photo we loaded earlier
inputs = blip_processor(content_image, return_tensors="pt").to("cuda")
output = blip_model.generate(**inputs, max_new_tokens=50)
caption = blip_processor.decode(output[0], skip_special_tokens=True)

plt.figure(figsize=(8, 6))
plt.imshow(content_image)
plt.axis('off')
plt.title(f"AI Caption: {caption}", fontsize=12, wrap=True)
plt.tight_layout()
plt.show()

print(f"Generated caption: {caption}")

### Conditional Captioning (Asking Questions About Images)

BLIP also supports **visual question answering**. You can ask specific questions about an architectural image.

In [ ]:
from transformers import BlipForQuestionAnswering

# Load the VQA model
blip_vqa = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to("cuda")

# Ask questions about the building
questions = [
    "What type of building is this?",
    "What materials can you see?",
    "How many floors does this building have?",
    "What is the architectural style?",
]

plt.figure(figsize=(8, 6))
plt.imshow(content_image)
plt.axis('off')
plt.title("Visual Question Answering", fontsize=14)
plt.show()

print("Questions and AI Answers:")
print("-" * 50)
for q in questions:
    inputs = blip_processor(content_image, q, return_tensors="pt").to("cuda")
    output = blip_vqa.generate(**inputs, max_new_tokens=20)
    answer = blip_processor.decode(output[0], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

Note that these models are general-purpose and not specifically trained on architecture, so their answers may be imprecise. Research groups are now working on **architecture-specific multimodal models** that better understand building typology, materials, and spatial relationships.

---

# Part 4: Large Language Models (LLMs) for Design Briefs

Large Language Models like GPT, Claude, and Gemini are being used in architecture for:

- **Design brief generation** from project constraints
- **Building program development** with space allocation
- **Material specification** and selection guidance
- **Code compliance** questions and review
- **Design critique** and feedback on proposals
- **Code generation** for parametric modeling scripts (Grasshopper, Dynamo, Rhino)

We will use a small, open-source language model to demonstrate how LLMs can generate design-related text. In practice, architects use larger models (GPT-4, Claude) through their web interfaces or APIs for higher quality output.

In [ ]:
from transformers import pipeline

# Load a small text generation model that runs on Colab
# For higher quality output in practice, use GPT-4, Claude, or Gemini
text_generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Text generation model loaded successfully.")

In [ ]:
# Generate a design brief from a short project description
prompt_text = """<|system|>
You are an architectural design assistant. Generate a concise design brief.
</s>
<|user|>
Generate a brief design program for a small community library in a residential neighborhood. Include the main spaces and approximate square footage for each.
</s>
<|assistant|>
"""

output = text_generator(
    prompt_text,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True,
)

generated_text = output[0]['generated_text']
# Extract only the assistant's response
response = generated_text.split("<|assistant|>")[-1].strip()
print("AI-Generated Design Brief:")
print("=" * 50)
print(response)

### Limitations of Small vs. Large Models

The small model above gives a rough demonstration, but larger models (GPT-4, Claude 3.5, Gemini) produce significantly better architectural content because they have been trained on a much larger corpus that includes architectural texts, building codes, and design literature.

**In practice, architects are using LLMs for:**

| Task | Example Prompt | Typical Model |
|---|---|---|
| Site analysis | "Summarize the zoning requirements for this parcel in Seattle" | GPT-4, Claude |
| Programming | "Generate a space program for a 50,000 SF middle school" | GPT-4, Claude |
| Precedent research | "What are notable examples of CLT construction in the Pacific Northwest?" | GPT-4 with web search |
| Code generation | "Write a Grasshopper Python script to generate a parametric facade" | GPT-4, Claude |
| Sustainability | "What passive cooling strategies are appropriate for a hot-arid climate?" | Any LLM |
| Client communication | "Draft a design narrative for this residential renovation" | GPT-4, Claude |

---

# Part 5: GANs and Floorplan Generation (Conceptual Overview)

**Generative Adversarial Networks (GANs)** have been one of the most researched generative AI techniques for architectural layout and floorplan generation. While training a GAN from scratch requires a large dataset and significant compute, understanding how they work is important for following current research.

### How GANs Work

A GAN consists of two neural networks competing against each other:

1. **Generator:** Takes random noise as input and produces a synthetic image (e.g., a floorplan)
2. **Discriminator:** Tries to distinguish between real floorplans (from the training set) and fake ones (from the generator)

During training, the generator learns to produce increasingly realistic outputs that fool the discriminator. The competition drives both networks to improve.

### Notable GAN-Based Architecture Research

| Paper / Project | What It Does |
|---|---|
| **House-GAN / House-GAN++** (Nauata et al.) | Generates floorplans from bubble diagrams and adjacency graphs |
| **ArchiGAN** (Chaillou, 2019) | Three-stage GAN: footprint, program layout, then furnished plan |
| **Graph2Plan** (Hu et al.) | Uses graph neural networks with GANs for constraint-aware layouts |
| **Pix2Pix for facades** (Isola et al.) | Translates facade sketches to photorealistic renderings |

### Beyond GANs: Newer Approaches

The field is moving beyond pure GAN-based generation toward:
- **Diffusion models** conditioned on spatial constraints
- **Tokenized floorplans** where layouts are treated as sequences (Qin et al., 2024)
- **Graph-based constraint optimization** combined with generative models (Goyal, 2024)
- **Hypergraph models** for space efficiency and carbon analysis (Weber et al.)

These newer approaches often produce more controllable and spatially coherent results than early GAN methods.

---

# Part 6: Challenges and Evaluation

### Current Limitations of Generative AI in Architecture

1. **Spatial incoherence** — AI-generated images may show impossible structural connections, floating elements, or inconsistent perspectives
2. **No building physics** — Generated images do not encode structural integrity, thermal performance, or building code compliance
3. **Style bias** — Models are biased toward the styles and geographies overrepresented in their training data
4. **Evaluation is difficult** — There are no widely agreed-upon metrics for measuring the architectural quality of AI-generated designs
5. **Authorship and ethics** — Questions about intellectual property, credit, and the role of the architect when AI generates the design

### How Are Researchers Evaluating AI-Generated Architecture?

| Method | What It Measures | Limitation |
|---|---|---|
| **FID Score** (Frechet Inception Distance) | Statistical similarity between generated and real images | Does not measure architectural quality |
| **User Studies** | Human judgment of realism, creativity, usefulness | Subjective, hard to reproduce |
| **Expert Review** | Architects evaluate functional and aesthetic qualities | Time-consuming, small sample sizes |
| **Performance Simulation** | Energy, daylight, structural analysis of generated designs | Only applicable to fully defined geometries |
| **Graph Matching** | How well generated floorplans match adjacency requirements | Specific to layout generation |

As Jang et al. (2025) note in their review, the field still lacks standardized evaluation methods, which makes it difficult to compare different approaches rigorously.

---

# Exercises

**Exercise 1:** Using the Stable Diffusion model in Part 1, generate three images of the **same building type** (e.g., a library) in three different **architectural styles** (e.g., brutalist, parametric, vernacular). Save or screenshot the results and compare them.

**Exercise 2:** In Part 2, apply style transfer using a **hand sketch** or **watercolor painting** as the style image and a 3D rendering or building photograph as the content. Discuss whether the output could be useful in a design presentation.

**Exercise 3:** Use the BLIP model in Part 3 to caption three different building photographs. Evaluate whether the AI descriptions are architecturally accurate. What does the model get right? What does it miss?

**Exercise 4:** In a markdown cell below, write a short reflection (150-200 words) on the following question:

> How might generative AI change the role of the architect in the design process? Consider both the opportunities (speed, exploration, accessibility) and the risks (deskilling, homogenization, loss of authorship).

---

## Summary

In this notebook we explored the landscape of generative AI in architecture:

1. **Text-to-image diffusion models** (Stable Diffusion) can generate photorealistic architectural visualizations from text prompts, with prompt engineering and guidance scale as key controls.
2. **Neural style transfer** can apply artistic styles to building photographs, useful for presentation and mood visualization.
3. **Image captioning and VQA models** (BLIP) can analyze and describe architectural images, though they lack domain-specific vocabulary.
4. **Large Language Models** can assist with design briefs, programming, code generation, and research, with larger models producing better results.
5. **GANs and graph-based methods** have driven floorplan generation research, though newer diffusion and token-based approaches are emerging.
6. **Evaluation remains a major challenge** — the field lacks standardized metrics for assessing AI-generated architectural designs.

### Key Takeaway

Generative AI in architecture is currently most useful in the **early design stages** — concept exploration, visual communication, and rapid ideation. As models become more spatially aware and physics-informed, they may eventually support later design stages including performance analysis and construction documentation.

### References

- Jang, S., Roh, H., & Lee, G. (2025). Generative AI in Architectural Design: Application, Data, and Evaluation Methods. *Automation in Construction*, 174.
- Cheung, L.H., Wang, L., & Lei, D. (2025). Conversational, Agentic AI-Enhanced Architectural Design Process. *Architectural Intelligence*, 4(10).
- Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks. *Journal of Computational Physics*.
- Nauata, N., Hosseini, S., Chang, K.H., Chu, H., Cheng, C.Y., & Furukawa, Y. (2021). House-GAN++. *CVPR*.
- Chaillou, S. (2019). ArchiGAN: A Generative Stack for Apartment Building Design. *ACADIA*.

---

# Assignment: Generative AI in Architecture

**Instructions:** Complete all the tasks below by writing and running code in the provided code cells. Add markdown cells to explain your observations where indicated. Submit the entire notebook (.ipynb) with all outputs visible.

> **Grading:** Each task is worth equal credit. Partial credit is given for reasonable attempts with explanations.


### Task 1: Generate a Building in Your Studio Project Context

Using the Stable Diffusion pipeline from Part 1, write a prompt that generates an image of a building related to your current studio project (or a building type of your choice). Your prompt must include at least **5 specific descriptors** (building type, material, context/site, lighting, style). Display the generated image with a title.


In [ ]:
# Task 1: Your code here


### Task 2: Seed Variation Study

Using the **same prompt** from Task 1, generate **4 different images** by changing only the random seed (use seeds 100, 200, 300, 400). Display all four images side by side using `show_images()`. In a markdown cell below your code, briefly describe how the outputs differ despite having the same prompt.


In [ ]:
# Task 2: Your code here


*Task 2 — Your observations:*



### Task 3: Guidance Scale Experiment

Pick a new architectural prompt of your choice. Generate images at guidance scales of **2.0, 5.0, 7.5, 10.0, and 15.0** (use the same seed for all). Display them side by side and write a short paragraph analyzing the visual quality and prompt adherence at each level.


In [ ]:
# Task 3: Your code here


*Task 3 — Your analysis:*



### Task 4: Prompt Engineering — Material Substitution

Write a base prompt describing a **pavilion** design. Then create **3 variations** where you only change the primary material (e.g., timber, steel and glass, rammed earth). Generate and display all three. Label each image with its material.


In [ ]:
# Task 4: Your code here


### Task 5: Style Transfer — Apply Two Different Styles to One Building

Load a building photograph (use a URL or upload your own). Apply neural style transfer using **two different style images** (e.g., a watercolor painting and an oil painting, or a sketch and a photograph of a material texture). Display the original content image and both stylized results side by side.


In [ ]:
# Task 5: Your code here


### Task 6: Style Transfer — Style an AI-Generated Image

Take one of the images you generated with Stable Diffusion (from any earlier task) and use it as the **content image** for neural style transfer. Choose a style image that represents an architectural rendering technique (e.g., charcoal sketch, section perspective, collage). Display the result and comment on whether the combination is useful for design communication.


In [ ]:
# Task 6: Your code here


*Task 6 — Your comments:*



### Task 7: Image Captioning — Compare Captions Across Building Types

Find or generate **3 images of different building types** (e.g., a skyscraper, a residential house, a religious building). Run BLIP captioning on each and display each image with its generated caption. Comment on the accuracy.


In [ ]:
# Task 7: Your code here


*Task 7 — Your comments on caption accuracy:*



### Task 8: Visual Question Answering — Interrogate a Building Image

Choose one building image (your own photo, a downloaded image, or an AI-generated one). Ask the BLIP VQA model at least **5 architecture-related questions** about it (e.g., "What is the roof shape?", "How many floors does this building have?", "What is the facade material?"). Print each question and answer. Comment on which answers are correct and which are wrong.


In [ ]:
# Task 8: Your code here


*Task 8 — Your evaluation of the VQA answers:*



### Task 9: Design Brief Generation with an LLM

Using the TinyLlama text generation pipeline from Part 4, generate a design brief for a building type of your choice. Customize the system prompt and user prompt to include specific site constraints (climate, neighborhood, lot size). Print the generated brief.


In [ ]:
# Task 9: Your code here


### Task 10: Compare Two LLM Prompts for Program Generation

Write **two different prompts** to the TinyLlama model asking it to generate a space program (list of rooms with approximate areas) for the **same building type**. One prompt should be vague and one should be highly detailed with constraints. Generate both outputs and compare them. Which prompt produced a more useful result?


In [ ]:
# Task 10: Your code here


*Task 10 — Your comparison:*



### Task 11: Negative Prompts

Stable Diffusion supports **negative prompts** — text describing what you do *not* want in the image. Generate the same architectural scene **with and without** a negative prompt (e.g., `negative_prompt="blurry, low quality, distorted, people"`). Display both results side by side and describe the differences.

**Hint:** Pass `negative_prompt=` as a keyword argument to `pipe()`.


In [ ]:
# Task 11: Your code here


*Task 11 — Your observations:*



### Task 12: Resolution and Aspect Ratio

Generate the same prompt at three different resolutions/aspect ratios: **512×512** (square), **512×768** (portrait), and **768×512** (landscape). Use the `height` and `width` parameters in `pipe()`. Display all three and discuss which aspect ratio works best for architectural visualization and why.


In [ ]:
# Task 12: Your code here


*Task 12 — Your discussion:*



### Task 13: Inference Steps Comparison

The `num_inference_steps` parameter controls the number of denoising steps in the diffusion process. Generate the same prompt with **10, 25, 50, and 75 steps** (same seed). Display all four results and describe the trade-off between quality and generation time.

**Hint:** Use Python's `time` module to measure how long each generation takes.


In [ ]:
# Task 13: Your code here


*Task 13 — Your analysis of the quality vs. time trade-off:*



### Task 14: Conditional Captioning as a Design Critic

Generate an architectural image with Stable Diffusion. Then use BLIP's **conditional captioning** feature to describe the image from different perspectives by providing different text prefixes. Use at least 3 prefixes, for example:
- `"This building is"`
- `"The architectural style is"`
- `"The main material is"`

Print each prefix and the completed caption. Discuss whether the model's descriptions match your design intent.


In [ ]:
# Task 14: Your code here


*Task 14 — Your discussion:*



### Task 15: Critical Reflection

In a **300–500 word** written response (in the markdown cell below), reflect on the following:

1. Which generative AI technique explored in this notebook do you find most useful for your own design process, and why?
2. What are the biggest limitations you encountered when using these tools for architectural design?
3. How might generative AI change the role of the architect in the next 5–10 years? What skills should architecture students develop to work effectively alongside these tools?

Support your response with specific examples from the tasks you completed above.


*Task 15 — Your reflection:*


